# 1. Verificação do motor de autodiff

Compara os gradientes do `backward()` com diferenças finitas: $\frac{f(x+\epsilon) - f(x-\epsilon)}{2\epsilon}$.

In [ ]:
import sys
sys.path.append("..")

import numpy as np

from src.autodiff import Tensor
from src.gradcheck import check_gradient
from src.nn import MLP
from src.losses import mse, bce

## Exemplo simples

$f(x, y) = x y + x^2$, então $\partial f/\partial x = y + 2x = 7$ e $\partial f/\partial y = x = 2$.

In [2]:
x = Tensor(2.0)
y = Tensor(3.0)

f = x * y + x ** 2
f.backward()

print("f     =", f.data)
print("df/dx =", x.grad)
print("df/dy =", y.grad)

f     = 10.0
df/dx = 7.0
df/dy = 2.0


## Todas as operações

`a` matriz, `b` vetor (testa broadcast), `s` escalar, `m` matriz para o produto matricial.

In [ ]:
rng = np.random.default_rng(0)

a = Tensor(rng.uniform(0.5, 2.0, (3, 4)))
b = Tensor(rng.uniform(0.5, 2.0, (4,)))
s = Tensor(1.7)
m = Tensor(rng.normal(size=(4, 2)))


def test_add():
    return (a + b + s + 2.0).sum()

def test_subtract():
    return (a - b - s).sum() + (3.0 - a).sum()

def test_multiply():
    return (a * b * s * 3.0).sum()

def test_divide():
    return (a / b / s).sum() + (2.0 / a).sum() + (s / b).sum()

def test_matmul():
    product = a @ m
    return (product * product).sum()

def test_power():
    return (a ** 3).sum() + (a ** b).sum() + (2.0 ** a).sum() + s ** 2.5

def test_root():
    return a.sqrt().sum() + b.root(3).sum()

def test_exp():
    return a.exp().sum() + s.exp()

def test_log():
    return a.log().sum() + b.log(base=10).sum() + s.log(base=2)

def test_sum():
    return (a.sum(axis=0) * b).sum() + (a.sum(axis=1, keepdims=True) * a).sum() + a.mean()

def test_activations():
    shifted = a - 1.2
    return (shifted.relu() + shifted.leaky_relu() + a.sigmoid() + a.tanh()).sum()


tests = [
    ("add", test_add),
    ("subtract", test_subtract),
    ("multiply", test_multiply),
    ("divide", test_divide),
    ("matmul", test_matmul),
    ("power", test_power),
    ("root", test_root),
    ("exp", test_exp),
    ("log", test_log),
    ("sum / mean", test_sum),
    ("activations", test_activations),
]

for name, function in tests:
    error = check_gradient(function, [a, b, s, m])
    print("%-12s largest error = %.2e" % (name, error))

add          largest error = 6.64e-09
subtract     largest error = 3.08e-09
multiply     largest error = 1.06e-08
divide       largest error = 3.59e-09
matmul       largest error = 4.17e-09
power        largest error = 6.94e-09
root         largest error = 2.53e-09
exp          largest error = 2.14e-09
log          largest error = 2.20e-10
sum / mean   largest error = 8.41e-09
activations  largest error = 1.69e-09


## MLP

Mesma verificação em todos os pesos de uma rede pequena, com BCE e MSE.

In [ ]:
model = MLP(input_size=4, hidden_sizes=[5, 5], output_size=3,
            hidden_activation="tanh", output_activation="sigmoid", init="xavier")

X = rng.normal(size=(6, 4))
y = (rng.uniform(size=(6, 3)) > 0.5) * 1.0


def mlp_with_bce():
    return bce(model.forward(X), y)

def mlp_with_mse():
    return mse(model.forward(X), y)


print("MLP + BCE largest error = %.2e" % check_gradient(mlp_with_bce, model.parameters()))
print("MLP + MSE largest error = %.2e" % check_gradient(mlp_with_mse, model.parameters()))

MLP + BCE  largest error = 1.53e-10
MLP + MSE  largest error = 3.37e-11


Erros de ~$10^{-8}$ ou menores: o backward do motor está correto.